# Epitope–paratope attention map

Where does the joint stack's attention land on the antigen, from the antibody's CDR3 positions? Needs `matplotlib` (not a core dependency of this package -- `pip install matplotlib` if you don't have it).

In [ ]:
import matplotlib.pyplot as plt
import langaai

model = langaai.load()

In [ ]:
heavy = "EVQLVESGGGLVQPGGSLRLSCAASGFNFKDTYIHWVRQAPGKGLEWVARIYPANGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCASDGGSYSYAFDYWGQGTLVTVSS"
light = "DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIK"
antigen = "TQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTNASLSFLQDIQEVQGYVLIAHNQVRQVPLQRLRIVRGTQLFEDNYALAVLDNGDPLNNTTPVTGASPGGLRELQLRSLTEILKGGVLIQRNPQLCYQDTILWKDIFHKNNQLALTLIDTNRSRACHPCSPMCKGSRCWGESSEDCQSLTRTVCAGGCARCKGPLPTDCCHEQCAAGCTGPKHSDCLACLHFNHSGICELHCPALVTYNTDTFESMPNPEGRYTFGASCVTACPYNYLSTDVGSCTLVCPLHNQEVTAEDGTQRCEKCSKPCARVCYGLGMEHLREVRAVTSANIQEFAGCKKIFGSLAFLPESFDGDPASNTAPLQPEQLQVFETLEEITGYLYISAWPDSLPDLSVFQNLQVIRGRILHNGAYSLTLQGLGISWLGLRSLRELGSGLALIHHNTHLCFVHTVPWDQLFRNPHQALLHTANRPEDECVGEGLACHQLCARGHCWGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYVNARHCLPCHPECQPQNGSVTCFGPEADQCVACAHYKDPPFCVARCPSGVKPDLSYMPIWKFPDEEGACQPCPIN"
print(len(heavy), len(light), len(antigen))

ab = model.encode_antibody(heavy, light)
ag = model.embed_antigen(antigen)
cdr3_start = heavy.find("ASDGGSYSYAFDY")
cdr3_span = slice(cdr3_start, cdr3_start + len("ASDGGSYSYAFDY"))
print("CDR3 token positions:", cdr3_span.start, "-", cdr3_span.stop)

`ab_to_ag` averaged over every layer and head, restricted to the CDR3 rows -- one attention profile over the antigen per CDR3 residue. 

In [ ]:
[block] = model.attention_block([(ab, ag)], "ab_to_ag", average=True)  # [n_ab, n_ag]
cdr3_to_ag = block[cdr3_span]  # [cdr3_len, n_ag]

fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(cdr3_to_ag.numpy(), aspect="auto", cmap="viridis")
ax.set_xlabel("antigen position")
ax.set_ylabel("CDR3 position")
ax.set_title("CDR3 -> antigen attention (averaged over layers/heads)")
fig.colorbar(im, ax=ax, label="attention weight")
fig.tight_layout()

Mean attention profile over the antigen, pooled across CDR3 positions -- which antigen residues does CDR3 attend to most, on average?

In [ ]:
mean_profile = cdr3_to_ag.mean(dim=0)
top_k = 10
top_positions = mean_profile.topk(top_k).indices.tolist()

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(mean_profile.numpy())
ax.set_xlabel("antigen position")
ax.set_ylabel("mean attention from CDR3")
ax.set_title(f"top {top_k} attended antigen positions: {sorted(top_positions)}")
fig.tight_layout()